In [1]:
from PreRun import PreRun
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as mse
from itertools import product
from datetime import date

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [3]:
params = {
    'objective': 'regression',
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metrics': 'mse',
    'shrinkage_rate': 0.1
}

In [4]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [5]:
start_stop_dict = {
    system_id: {
        name[0]: [0, 0] for name in name_val
    } for system_id in systems_good_timezones_manual_edit
}
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        start_stop_dict[system_id].pop(name)
        continue
    if val == 'None':
        real_val = None
    else:
        real_val = val
    prerun_system = PreRun(system_id, f'./test_results/{system_id}-{val}/', real_val, systems_cleaned)
    start_stop_dict[system_id][name][0] = prerun_system.data.at[0, 'time'].date()
    last_index = prerun_system.data.index[-1]
    start_stop_dict[system_id][name][1] = prerun_system.data.at[last_index, 'time'].date()

In [6]:
start_stop_dict

{4: {'other': [datetime.date(2007, 9, 1), datetime.date(2023, 2, 28)]},
 10: {'other': [datetime.date(2006, 1, 25), datetime.date(2023, 2, 28)]},
 33: {'other': [datetime.date(2010, 11, 10), datetime.date(2023, 2, 28)]},
 36: {'other': [datetime.date(2012, 3, 30), datetime.date(2019, 7, 21)]},
 50: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 51: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 1199: {'inverter': [datetime.date(2010, 5, 29), datetime.date(2018, 8, 3)]},
 1204: {'inverter': [datetime.date(2011, 2, 9), datetime.date(2015, 1, 5)]},
 1283: {'inverter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)],
  'meter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)]},
 1284: {'other': [datetime.date(2012, 6, 30), datetime.date(2015, 3, 15)]},
 1289: {'other': [datetime.date(2012, 9, 28), datetime.date(2020, 5, 13)]},
 1332: {'inverter': [datetime.date(2013, 3, 30), datetime.date(2014, 7, 31)],
  'meter': [datetime.date(

In [14]:
def basic_lightbgm_test(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, params: dict):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=1, include_last_year=True, todays_lags=1, include_hour_cyclic=True,include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    df = prerun_system.amended_data
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '1_days_ago', '1_hours_ago_today', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_starter_train = df[df['time'] < pd.Timestamp(year=2018, month=1, day=1, hour=0)]
    X_starter_train = df_starter_train[my_cols]
    y_starter_train = df_starter_train['energy']
    lgb_tr = lgb.Dataset(X_starter_train, label=y_starter_train)
    df_starter_val = df[(df['time'] >= pd.Timestamp(year=2019, month=1, day=1, hour=0))
                        & (df['time'] < pd.Timestamp(year=2021, month=1, day=1, hour=0))]
    X_val = df_starter_val[my_cols]
    y_val = df_starter_val['energy']
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_tr)
    df_starter_test = df[df['time'] >= pd.Timestamp(year=2022, month=1, day=1, hour=0)]
    X_test = df_starter_test[my_cols]
    y_test = df_starter_test['energy']
    num_round = 100
    bst = lgb.train(params, lgb_tr, num_round, valid_sets=[lgb_val,])
    y_pred = bst.predict(X_test)
    print(f'Mean squared error: {mse(y_test, y_pred):.5f}')
    

In [15]:
basic_lightbgm_test(4, './test_results/4-None', None, systems_cleaned, params)

                     time    energy
0     2007-09-01 06:00:00  0.032494
1     2007-09-01 07:00:00  0.159633
2     2007-09-01 08:00:00  0.304443
3     2007-09-01 09:00:00  0.265836
4     2007-09-01 10:00:00  0.804245
...                   ...       ...
43271 2023-02-28 13:00:00  0.610406
43272 2023-02-28 14:00:00  0.408742
43273 2023-02-28 15:00:00  0.395145
43274 2023-02-28 16:00:00  0.230329
43275 2023-02-28 17:00:00  0.003725

[43276 rows x 2 columns]
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000752 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [Warning] 

In [7]:
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [8]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))

In [9]:
from itertools import product

In [21]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    print(basic_lightbgm_test(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned, params))

4-other
                     time    energy
0     2007-09-01 06:00:00  0.032494
1     2007-09-01 07:00:00  0.159633
2     2007-09-01 08:00:00  0.304443
3     2007-09-01 09:00:00  0.265836
4     2007-09-01 10:00:00  0.804245
...                   ...       ...
43271 2023-02-28 13:00:00  0.610406
43272 2023-02-28 14:00:00  0.408742
43273 2023-02-28 15:00:00  0.395145
43274 2023-02-28 16:00:00  0.230329
43275 2023-02-28 17:00:00  0.003725

[43276 rows x 2 columns]
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000678 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [W

ValueError: Input data must be 2 dimensional and non empty.

In [15]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    trial_data = PreRun(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned)
    trial_data.good_end_days_naive(streak=7)
    num_good_days = len(trial_data.good_days)
    num_good_ends = len(trial_data.end_days_naive)
    the_pct = num_good_ends / num_good_days * 100
    print(f'Good ends count: {num_good_ends}')
    print(f'Good days count: {num_good_days}')
    print(f'Ratio: {the_pct:.5f}%')
    print('')


4-other
Good ends count: 1127
Good days count: 3598
Ratio: 31.32296%

10-other
Good ends count: 891
Good days count: 4170
Ratio: 21.36691%

33-other
Good ends count: 1484
Good days count: 3594
Ratio: 41.29104%

36-other
Good ends count: 358
Good days count: 1510
Ratio: 23.70861%

50-other
Good ends count: 2062
Good days count: 5817
Ratio: 35.44783%

51-other
Good ends count: 1784
Good days count: 5644
Ratio: 31.60879%

1199-inverter
Good ends count: 828
Good days count: 2275
Ratio: 36.39560%

1204-inverter
Good ends count: 428
Good days count: 1164
Ratio: 36.76976%

1283-inverter
Good ends count: 857
Good days count: 2124
Ratio: 40.34840%

1283-meter
Good ends count: 864
Good days count: 2136
Ratio: 40.44944%

1284-other
Good ends count: 177
Good days count: 746
Ratio: 23.72654%

1289-other
Good ends count: 322
Good days count: 1774
Ratio: 18.15107%

1332-inverter
Good ends count: 110
Good days count: 310
Ratio: 35.48387%

1332-meter
Good ends count: 937
Good days count: 2185
Ratio: 42

In [16]:
shorter_test = PreRun(4903, f'./test_results/4903-inverter', 'inverter', systems_cleaned)
shorter_test.add_weather_features_only()

In [30]:
shorter_test.good_days.iloc[30:40]

,date
30,2014-09-23
31,2014-09-24
32,2014-09-26
33,2014-09-27
34,2014-09-28
35,2014-09-29
36,2014-09-30
37,2014-10-01
38,2014-10-02
39,2014-10-03


In [18]:
shorter_test.good_end_days_naive(streak=7)

,date
0,2014-09-11
1,2014-09-12
2,2014-09-13
3,2014-09-14
4,2014-09-15
...,...
422,2017-12-06
423,2017-12-07
424,2017-12-08
425,2017-12-22


In [34]:
good_days_ext = shorter_test.good_days
end_days_ext = shorter_test.end_days_naive

In [35]:
end_days_ext['diff'] = end_days_ext['date'].diff(periods=1)

In [43]:
good_days_ext['diff'] = good_days_ext['date'].diff(periods=1)

In [50]:
good_days_ext.iloc[32:52]

,date,diff
32,2014-09-26,2 days
33,2014-09-27,1 days
34,2014-09-28,1 days
35,2014-09-29,1 days
36,2014-09-30,1 days
37,2014-10-01,1 days
38,2014-10-02,1 days
39,2014-10-03,1 days
40,2014-10-04,1 days
41,2014-10-05,1 days


In [49]:
end_days_ext.iloc[5:21]

,date,diff
5,2014-09-16,1 days
6,2014-09-24,8 days
7,2014-10-02,8 days
8,2014-10-03,1 days
9,2014-10-04,1 days
10,2014-10-05,1 days
11,2014-10-06,1 days
12,2014-10-07,1 days
13,2014-10-08,1 days
14,2014-10-09,1 days


In [32]:
shorter_test.end_days_naive.diff(periods=1).iloc[10:20]

,date
10,1 days
11,1 days
12,1 days
13,1 days
14,1 days
15,1 days
16,1 days
17,1 days
18,1 days
19,1 days
